# Student Performance Data Analysis Mini-Project

This notebook performs an end-to-end data analysis of student performance, including:
1. **Dataset Generation & Exploration**
2. **Data Cleaning & Feature Engineering**
3. **Statistical Analysis & Insights**
4. **Data Visualization**

### 1. Imports & Dataset Generation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Create a Sample Dataset (Replace with pd.read_csv() if using your own CSV)
np.random.seed(42)
num_students = 60

data = {
    "Student_ID": [f"STU{i:03d}" for i in range(1, num_students + 1)],
    "Gender": np.random.choice(["Male", "Female"], size=num_students),
    "Study_Hours_Per_Day": np.round(np.random.uniform(1.0, 6.5, size=num_students), 1),
    "Attendance_Pct": np.round(np.random.uniform(55.0, 100.0, size=num_students), 1),
    "Math_Score": np.clip(np.random.normal(68, 15, size=num_students).round(0), 25, 100),
    "Science_Score": np.clip(np.random.normal(71, 14, size=num_students).round(0), 30, 100),
    "English_Score": np.clip(np.random.normal(74, 12, size=num_students).round(0), 35, 100),
}

df = pd.DataFrame(data)

# Introduce a couple of missing values to demonstrate data cleaning
df.loc[3, "Math_Score"] = np.nan
df.loc[12, "Attendance_Pct"] = np.nan

# Save raw dataset to CSV
df.to_csv("student_data_raw.csv", index=False)

print("Dataset created successfully with missing values introduced!")
df.head()

### 2. Data Cleaning & Feature Engineering

In [ ]:
print("=== DATA CLEANING ===")
print("Missing values before cleaning:\n", df.isnull().sum(), "\n")

# Fill missing numeric values with the column median
df["Math_Score"] = df["Math_Score"].fillna(df["Math_Score"].median())
df["Attendance_Pct"] = df["Attendance_Pct"].fillna(df["Attendance_Pct"].median())

# Calculate Total and Average Score
subjects = ["Math_Score", "Science_Score", "English_Score"]
df["Total_Score"] = df[subjects].sum(axis=1)
df["Average_Score"] = df[subjects].mean(axis=1).round(2)

# Categorize performance into grades/tiers
def categorize_performance(avg):
    if avg >= 80:
        return "Distinction"
    elif avg >= 60:
        return "Good"
    elif avg >= 40:
        return "Average"
    else:
        return "At Risk"

df["Performance_Category"] = df["Average_Score"].apply(categorize_performance)
df["Result"] = np.where(df["Average_Score"] >= 40, "Pass", "Fail")

print("Data cleaning and feature engineering complete!")
df.info()

### 3. Statistical Analysis & Insights

In [ ]:
print("=== SUMMARY STATISTICS ===")
print(df[["Study_Hours_Per_Day", "Attendance_Pct", "Average_Score"]].describe().round(2), "\n")

print("=== PERFORMANCE CATEGORY COUNTS ===")
print(df["Performance_Category"].value_counts(), "\n")

print("=== AVERAGE SCORE BY GENDER ===")
print(df.groupby("Gender")[subjects + ["Average_Score"]].mean().round(2), "\n")

print("=== CORRELATION WITH AVERAGE SCORE ===")
numeric_cols = ["Study_Hours_Per_Day", "Attendance_Pct", "Math_Score", "Science_Score", "English_Score", "Average_Score"]
print(df[numeric_cols].corr()["Average_Score"].sort_values(ascending=False).round(2), "\n")

top_students = df.sort_values(by="Average_Score", ascending=False).head(5)
print("=== TOP 5 STUDENTS ===")
print(top_students[["Student_ID", "Average_Score", "Attendance_Pct", "Performance_Category"]].to_string(index=False))

# Save processed dataset
df.to_csv("student_data_processed.csv", index=False)

### 4. Data Visualization

In [ ]:
# Chart 1: Subject-wise Average Scores
subject_means = df[subjects].mean()
plt.figure(figsize=(7, 4.5))
plt.bar(["Math", "Science", "English"], subject_means.values, color=["#4C72B0", "#55A868", "#C44E52"])
plt.title("Average Score by Subject")
plt.ylabel("Average Score")
plt.ylim(0, 100)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("subject_averages.png")
plt.show()

# Chart 2: Study Hours vs. Average Score
plt.figure(figsize=(7, 4.5))
plt.scatter(df["Study_Hours_Per_Day"], df["Average_Score"], alpha=0.75, color="#2b5c8f")
plt.title("Daily Study Hours vs. Average Score")
plt.xlabel("Study Hours Per Day")
plt.ylabel("Average Score")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("study_vs_score.png")
plt.show()